# Module Import Smoke Test

Run this notebook after installing the repository with `pip install -e .`. It checks that the structured modules import correctly and that the model/preprocessing helpers work on dummy data.

In [ ]:
import numpy as np
import torch

from bv_repro.models import ResNet18Gray, ResNetCrystal
from bv_repro.data import extract_vector
from bv_repro.preprocessing import normalize_nn_image, resize_image
from bv_repro.training import BenchmarkConfig, apply_crystal_mask, crystallographic_loss_v2
from bv_repro.visualization import plot_uncertainty_distributions

print("imports OK")


In [ ]:
config = BenchmarkConfig(epochs=1, pretrained=False)
print(config)

assert extract_vector("sample_b_-101_n_1-1-1.npy", "b").tolist() == [-1.0, 0.0, 1.0]
print("filename parsing OK")


In [ ]:
num_classes = 6
x = torch.rand(2, 1, 510, 170)

data_model = ResNet18Gray(num_classes=num_classes, pretrained=False)
crystal_model = ResNetCrystal(num_classes=num_classes, pretrained=False)

with torch.no_grad():
    data_logits = data_model(x)
    crystal_logits = crystal_model(x)

assert data_logits.shape == (2, num_classes)
assert crystal_logits.shape == (2, num_classes)
print("model forward pass OK", data_logits.shape, crystal_logits.shape)


In [ ]:
valid_mask = torch.tensor([
    [1, 1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0, 1],
], dtype=torch.float32)
targets = torch.tensor([0, 2], dtype=torch.long)

masked_logits = apply_crystal_mask(crystal_logits, valid_mask)
loss, ce, invalid, margin = crystallographic_loss_v2(
    crystal_logits,
    targets,
    valid_mask,
)

assert torch.isfinite(loss)
print("crystallographic loss OK", float(loss), float(ce), float(invalid), float(margin))


In [ ]:
image = np.random.default_rng(0).random((64, 32)).astype(np.float32)
resized = resize_image(image, target_shape=(510, 170))
normalized = normalize_nn_image(resized)

assert resized.shape == (510, 170)
assert normalized.shape == (510, 170)
assert np.isfinite(normalized).all()
print("preprocessing OK", normalized.shape, float(normalized.min()), float(normalized.max()))
